## Environment

In [1]:
from datetime import datetime
from pathlib import Path
import os
import sys

import pandas as pd

NOTE_DIR = Path.cwd().resolve().parents[3] / 'note'
REPO_ROOT = Path('/Users/xinc/GitHub/Quant')
sys.path.extend([str(NOTE_DIR), str(REPO_ROOT), os.getcwd()])

%load_ext autoreload
%autoreload 2

from module.get_info_FinMind import FinMindClient
from module.get_info_TradingView import Interval, TradingViewClient
from module.get_info_TWSE import GetInfoTWSE
from module.options.option_tools import compute_iv
from analyzer import StrategyConfig, TXAnalyzer
from cloud_data import TW_OPTIONS_SETTLE_TXO

START = '2020-01-01'
END = datetime.now().strftime('%Y-%m-%d')

TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.05
SETTLEMENT_PATH = TW_OPTIONS_SETTLE_TXO

tv = TradingViewClient()
fm = FinMindClient()
twse = GetInfoTWSE()

## Base Market Data

In [2]:
fm.initialize_frame(stock_id='TX', start_time=START, end_time=END)
analyzer = TXAnalyzer(fm.get_future_price())
analyzer.session_alignment_report()

split_dates = TXAnalyzer.split_periods(
    analyzer.display_df().index,
    start=StrategyConfig().backtest_start_date,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
)
TRAIN_END = split_dates['train_end']
VALIDATION_START = split_dates['validation_start']
VALIDATION_END = split_dates['validation_end']
TEST_START = split_dates['test_start']
pd.Series(split_dates, name='date')

[LOCAL] finmind/future_price -> /Users/xinc/GitHub/google_drive/Data/trading/tw_futures/TX.csv


train_end          2026-06-05
validation_start   2026-06-08
validation_end     2026-06-11
test_start         2026-06-12
Name: date, dtype: datetime64[ns]

## Factor Discovery

每個小節都是完整的探索單元：取得資料、對齊資料、畫圖或檢查關係。Discovery 一律只使用 training 資料，不會查看 validation 或 test 期間。


### Price and Calendar

In [7]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.daily_ret()
training_analyzer.monthly_ret(mode='benchmark')
training_analyzer.indicator_weekday_stats()
training_analyzer.indicator_ma_divergence(window=25)
training_analyzer.indicator_hist_vol(window=40)
training_analyzer.indicator_night_price()
training_analyzer.indicator_gap_days(after_holiday=False)
training_analyzer.indicator_gap_days(after_holiday=True)

#### 深度檢測（日盤） ma divergence

In [13]:
training_analyzer.compare_ma_divergence_windows(
    range(5, 51, 1),
    bin_percentile=4,
    demean_return=False,
)

In [55]:
report = training_analyzer.compare_ma_divergence_by_volatility(
    range(5, 51, 1),
    return_column='daily_ret',
    volatility_window=20,
    volatility_bins=5,
    divergence_bin_percentile=15,
)

volatility_report = training_analyzer.compare_ma_divergence_by_volatility(
    [25],
    return_column='daily_ret',
    vary='volatility',
    volatility_windows=range(5, 51, 1),
    volatility_bins=5,
    divergence_bin_percentile=15,
)

In [56]:
overlap_events = training_analyzer.show_divergence_signal_overlap([
    {
        'ma_window': 25,
        'volatility_window': 21,
        'volatility_regime': 4,
        'volatility_bins': 5,
        'divergence_percentile': 15,
    },
    {
        'ma_window': 25,
        'volatility_window': 40,
        'volatility_regime': 4,
        'volatility_bins': 5,
        'divergence_percentile': 15,
    },
])

In [57]:
signal_events = training_analyzer.show_divergence_signal_timeline(
    ma_window=25,
    volatility_window=40,
    volatility_regime=4,
    volatility_bins=5,
    divergence_percentile=15,
)

#### 深度檢測（夜盤） ma_divergence x hist_vol

### Option Positioning

In [ ]:
raw_day = fm.option_institution_position('TXO', start_date=START, end_date=END).copy()
raw_night = twse.get_institution_option_position(
    trading_session='night',
    start_time=START,
    end_time=END,
).copy()
analyzer.add_option_signals(raw_day, raw_night)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_opt_position(
    indicator='Foreign_Opt_Signal',
    trading_session='day',
    sub_analysis=False,
)

### Option Implied Volatility

In [ ]:
option_daily = TXAnalyzer.fetch_option_daily(
    fm,
    option_id='TXO',
    start_date=START,
    end_date=END,
)
settlement_df = pd.read_csv(SETTLEMENT_PATH)
analyzer.add_option_iv_skew(
    option_daily,
    settlement_df,
    iv_calculator=compute_iv,
    risk_free_rate=0.015,
)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_option_iv(trading_session='day')

### Sentiment

In [ ]:
historical_fear_greed = fm.get_cnn_fear_greed(START, END)
latest_fear_greed = TXAnalyzer.fetch_fear_greed()
analyzer.add_fear_greed(historical_fear_greed, latest_fear_greed)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_fear_greed(trading_session='day')
training_analyzer.indicator_fear_greed(trading_session='night')

### Global Market Factors

In [ ]:
move = tv.get_hist(
    symbol='MOVE',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(move, 'MOVE')
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_move(trading_session='day')
training_analyzer.indicator_move(trading_session='night')
training_analyzer.indicator_move(trading_session='day', sub_analysis=True)

sox = tv.get_hist(
    symbol='SOX',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(sox, 'SOX')
training_analyzer.indicator_sox(trading_session='day')

### Rates and Margin

In [ ]:
bond_5_year = fm.get_US_bond('United States 5-Year', START, END)
bond_features = bond_5_year[['date', 'value']].rename(columns={'value': 'US_bond_5y'}).set_index('date')
analyzer.merge_features(bond_features)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_US_bond(indicator='yield_shock')

margin_maintenance = fm.get_total_margin_maintenance(start_time=START, end_time=END)
analyzer.merge_features(margin_maintenance.set_index('date'))
margin_balance = fm.get_total_margin_info()
margin_features = margin_balance.pivot_table(index='date', columns='name', values='TodayBalance')
analyzer.merge_features(margin_features)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_maintenance_rate(point_version=False)
training_analyzer.indicator_margin_delta()

## Research Data Check

確認這次實際跑過的研究資料是否已合併、是否有缺值，以及最後更新日。

In [ ]:
analyzer.for_period(end=TRAIN_END).feature_status([
    'MOVE_open',
    'SOX_open',
    'Foreign_Opt_Signal_a',
    'SkewSlope',
    'fear_greed',
    'US_bond_5y',
])

## Hypothesis and Position Design

只有通過探索、且能說明交易理由的關係才放進設定。這裡建立基準假設與要驗證的候選部位。

In [ ]:
baseline_config = StrategyConfig()

candidate_config = StrategyConfig(
    day_long_position=0.5,
    day_short_position=-0.5,
    night_position=0.0,
)

## Validation

只用 validation 期間比較已經提出的設定，選定一個設定後就不要再依 validation 或 test 結果修改它。


In [ ]:
baseline_validation = analyzer.evaluate(
    baseline_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)
candidate_validation = analyzer.evaluate(
    candidate_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)

validation_summary = pd.concat(
    {
        'baseline': analyzer.summarize_result(baseline_validation),
        'candidate': analyzer.summarize_result(candidate_validation),
    },
    axis=1,
)
validation_summary


In [ ]:
validation_comparison = pd.DataFrame({
    'baseline': baseline_validation['strat_ret'],
    'candidate': candidate_validation['strat_ret'],
}).dropna()
validation_comparison.cumsum().plot(title='Validation: Baseline vs Candidate')


## Test

選定設定後才執行這段。Test 期間只用來確認固定策略的表現，不再調整設定。


In [ ]:
selected_config = baseline_config

test_result = analyzer.evaluate(selected_config, start=TEST_START)
test_summary = analyzer.summarize_result(test_result)
test_summary

# 確認 test 後，再將同一份固定設定寫入正式結果檔。
analyzer.backtest(config=selected_config, point_version=False, start=TEST_START)
